# 01 — 射频电磁场测试与可视化

对应 Rust 测试：`cargo test test_field_to_csv -- --nocapture`。读取 `results/fields/rf_field_test.csv`，显示 13.56 MHz 射频电场随时间的变化，并用解析公式检查结果。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 从 Jupyter 当前工作目录向上搜索 Cargo.toml，定位项目根目录
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / 'Cargo.toml').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('未找到 Cargo.toml，请从 Rust 项目下启动 Jupyter')
print('项目根目录:', PROJECT_ROOT)

In [ ]:
csv_path = PROJECT_ROOT / 'results' / 'fields' / 'rf_field_test.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'缺少 {csv_path}，先运行 cargo test test_field_to_csv -- --nocapture')
df = pd.read_csv(csv_path)
display(df.head())
print('数据点数:', len(df))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['time_ns'], df['Ex_V_m'], linewidth=1.5, label='Ex (13.56 MHz)')
ax.set(xlabel='Time (ns)', ylabel='Electric field (V/m)', title='RF electric field vs time')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df['time_ns'], df['Bz_T'] * 1e4, label='Bz', linewidth=1.5)
ax.set(xlabel='Time (ns)', ylabel='Magnetic field (G)', title='Static magnetic field vs time')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# 与解析模型比较，而不是把拟合曲线当作独立的实验结果
E0, f = 1000.0, 13.56e6
t = df['time_s'].to_numpy()
e_rust = df['Ex_V_m'].to_numpy()
e_analytical = E0 * np.cos(2 * np.pi * f * t)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t * 1e9, e_rust, label='Rust CSV', lw=2)
ax.plot(t * 1e9, e_analytical, '--', label='Analytical', lw=1.3)
ax.set(xlabel='Time (ns)', ylabel='Ex (V/m)', title='Rust result vs analytical field')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
print('最大绝对偏差 (V/m):', np.max(np.abs(e_rust - e_analytical)))